# 01 — Build the FDA novel-approval event list (2015–2025)

**Goal:** collect every CDER novel drug approval, attach the sponsor company, flag obvious large caps, and export a sheet for manual review.

**Why automate:** ~500 drugs is too many to type by hand, and a script is *reproducible*: anyone can rerun it and get the same raw list. That matters for credibility.

Run cells top to bottom with Shift+Enter.

In [7]:
import io, re, time
import requests
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0 (student research project)"}
YEARS = range(2021, 2026)
URL_PATTERNS = [
    "https://www.fda.gov/drugs/novel-drug-approvals-fda/novel-drug-approvals-{year}",
    "https://www.fda.gov/drugs/new-drugs-fda-cders-new-molecular-entities-and-new-therapeutic-biological-products/novel-drug-approvals-{year}",
]

## Step 1: Scrape the FDA yearly tables
FDA has used different URLs over the years, so we try two patterns. `pd.read_html` turns every HTML table on the page into a DataFrame, and we keep the one with a drug name and a date column.

In [8]:
def fetch_year(year):
    for pattern in URL_PATTERNS:
        url = pattern.format(year=year)
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code != 200:
                continue
            tables = pd.read_html(io.StringIO(r.text))
        except (requests.RequestException, ValueError):
            continue
        for t in tables:
            cols = [str(c).lower() for c in t.columns]
            if any("drug name" in c for c in cols) and any("date" in c for c in cols):
                t = t.copy()
                t["fda_year"] = year
                t["source_url"] = url
                return t
    return None

def standardize(t):
    rename = {}
    for c in t.columns:
        lc = str(c).lower()
        if "drug name" in lc:
            rename[c] = "drug_name"
        elif "active ingredient" in lc:
            rename[c] = "active_ingredient"
        elif "date" in lc:
            rename[c] = "fda_approval_date"
        elif "approved use" in lc:
            rename[c] = "approved_use"
    t = t.rename(columns=rename)
    keep = ["drug_name", "active_ingredient", "fda_approval_date", "approved_use", "fda_year", "source_url"]
    return t[[c for c in keep if c in t.columns]]

frames, failed = [], []
for y in YEARS:
    t = fetch_year(y)
    if t is None:
        failed.append(y)
        print(f"{y}: FAILED - handle manually")
    else:
        frames.append(standardize(t))
        print(f"{y}: {len(t)} approvals")
    time.sleep(1)  # be polite to fda.gov

events = pd.concat(frames, ignore_index=True)
print("Total:", len(events), "| Failed years:", failed)

2021: 50 approvals
2022: 37 approvals
2023: 55 approvals
2024: 50 approvals
2025: 46 approvals
Total: 238 | Failed years: []


In [5]:
path = "../data/raw/fda_nme_compilation_1985_2025.xlsx"
print(pd.ExcelFile(path).sheet_names)

nme = pd.read_excel(path)
print(nme.shape)
print(list(nme.columns))
nme.head()

['2025 Compilation']
(1387, 27)
['Proprietary  Name', 'Active Ingredient/Moiety', 'Applicant', 'NDA/BLA', ' Application Number(1)', ' Application Number(2)', ' Application Number(3)', 'Dosage Form(1)', 'Route of Administration(1)', 'Dosage Form(2)', 'Route of Administration(2)', 'Dosage Form(3)', 'Route of Administration(3)', 'FDA Receipt Date', 'FDA Approval Date', 'Approval Year', 'Abbreviated Indication(s)', 'Approved Use(s)', 'Review Designation', 'Orphan Drug Designation', 'Accelerated Approval', 'Breakthrough Therapy Designation', 'Fast Track Designation', 'Qualified Infectious Disease Product', 'Issued a Priority Review Voucher', 'Redeemed a Priority Review Voucher', 'Notes']


,Proprietary Name,Active Ingredient/Moiety,Applicant,NDA/BLA,Application Number(1),Application Number(2),Application Number(3),Dosage Form(1),Route of Administration(1),Dosage Form(2),...,Approved Use(s),Review Designation,Orphan Drug Designation,Accelerated Approval,Breakthrough Therapy Designation,Fast Track Designation,Qualified Infectious Disease Product,Issued a Priority Review Voucher,Redeemed a Priority Review Voucher,Notes
0,Lupron,leuprolide acetate,TAP Pharmaceuticals,NDA,19010,NaN,NaN,Injectable,Injection,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Seldane,terfenadine,Merrell-Dow,NDA,18949,NaN,NaN,Tablet,Oral,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ridaura,auranofin,SmithKline & French,NDA,18689,NaN,NaN,Capsule,Oral,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Marinol,dronabinol,Unimed,NDA,18651,NaN,NaN,Capsule,Oral,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fortaz,ceftazidime,Glaxo,NDA,50578,NaN,NaN,Injectable,Injection,NaN,...,NaN,Priority,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
nme.columns = nme.columns.str.strip().str.replace(r"\s+", " ", regex=True)
nme["FDA Approval Date"] = pd.to_datetime(nme["FDA Approval Date"], errors="coerce")

sample = nme[nme["Approval Year"].between(2015,2025)].copy()
print(len(sample), "approvals from 2015-2025")
print(sample.groupby("Approval Year").size())
sample.to_csv("../data/processed/nme_2015_2025.csv", index=False)

511 approvals from 2015-2025
Approval Year
2015    45
2016    22
2017    46
2018    59
2019    48
2020    53
2021    50
2022    37
2023    55
2024    50
2025    46
dtype: int64


In [9]:
print(sample["Applicant"].nunique(), "unique applicants")
sample["Applicant"].value_counts().head(40)

352 unique applicants


Applicant
Novartis Pharmaceuticals                      11
Pfizer                                         7
Eli Lilly                                      7
Novartis Pharmaceuticals Corporation           7
Genentech, Inc.                                7
Pfizer, Inc.                                   7
Genentech                                      6
Amgen                                          5
Janssen Biotech, Inc.                          5
Eli Lilly and Company                          5
Gilead Sciences                                4
AstraZeneca Pharmaceuticals                    4
Janssen Biotech                                4
Merck                                          4
AbbVie                                         4
Shionogi                                       4
Merck Sharp & Dohme                            4
Bayer HealthCare Pharmaceuticals, Inc.         4
Regeneron Pharmaceuticals, Inc.                4
AstraZeneca AB                                 4
Genzyme Co

In [13]:
LARGE = ["novartis", "pfizer", "lilly", "genentech", "roche", "amgen", "janssen",
         "johnson", "gilead", "astrazeneca", "merck", "abbvie", "bayer", "regeneron",
         "genzyme", "sanofi", "bristol", "squibb", "celgene", "vertex", "alexion",
         "novo nordisk", "glaxo", "boehringer", "biogen", "takeda", "shire",
         "allergan", "teva", "mylan", "alcon", "forest", "millennium", "viiv", "acerta",
         "ge healthcare", "genmab"]
NOT_US = ["shionogi", "daiichi", "astellas", "eisai", "otsuka", "ucb", "chugai",
          "emd serono", "csl behring", "fresenius", "kyowa", "mitsubishi",
          "nippon shinyaku", "santen", "taiho", "ipsen", "servier", "chiesi",
          "domp", "lundbeck", "leo pharma", "helsinn", "idorsia", "basilea",
          "biocodex", "bioprojet", "guerbet", "bracco", "blue earth", "medac",
          "hugel", "dizal", "akeso", "clinuvel", "botanix", "cosmo tech",
          "cassiopea", "d.o.r.c", "chemo research", "aop orphan", "acacia pharma",
          "evive", "italfarmaco", "novimmune", "oncopeptides", "santhera",
          "sk life", "sun pharmaceutical", "pharmaessentia", "taimed",
          "intrabio", "galderma", "ferrer"]
NOT_PUBLIC = ["drugs for neglected", "global alliance", "global health",
              "feinstein", "population council", "university",
              "us world meds", "amivas", "marathon pharma", "sprout"]

def screen(applicant):
    name = str(applicant).lower()
    if any(k in name for k in LARGE):
        return "exclude: large cap"
    if any(k in name for k in NOT_US):
        return "exclude: not US-listed"
    if any(k in name for k in NOT_PUBLIC):
        return "exclude: not a public company"
    return "review"

sample["screen"] = sample["Applicant"].apply(screen)
print(sample["screen"].value_counts())

screen
review                           226
exclude: large cap               193
exclude: not US-listed            81
exclude: not a public company     11
Name: count, dtype: int64


In [14]:
excluded = sample[sample["screen"] != "review"]
for reason, group in excluded.groupby("screen"):
    print(reason)
    print(sorted(group["Applicant"].unique()))
    print()

exclude: large cap
['AbbVie', 'AbbVie, Inc.', 'Abbvie', 'Acerta Pharma B.V.', 'Alcon Laboratories, Inc.', 'Alexion Pharmaceuticals', 'Alexion Pharmaceuticals, Inc.', 'Allergan', 'Amgen', 'Amgen, Inc.', 'AstraZeneca AB', 'AstraZeneca Pharmaceuticals', 'AstraZeneca Pharmaceuticals LP', 'AstraZeneca UK Limited', 'Bayer HealthCare Pharmaceuticals', 'Bayer HealthCare Pharmaceuticals, Inc.', 'Biogen', 'Biogen, Inc.', 'Boehringer Ingelheim Pharmaceuticals', 'Boehringer Ingelheim Pharmaceuticals, Inc.', 'Bristol-Myers Squibb', 'Bristol-Myers Squibb Company', 'Celgene', 'Celgene Corporation', 'Eli Lilly', 'Eli Lilly and Company', 'Forest Laboratories', 'Forest Research Institute', 'GE Healthcare, Inc.', 'Genentech', 'Genentech, Inc.', 'Genmab US, Inc.', 'Genzyme Corporation', 'Gilead Sciences', 'Gilead Sciences, Inc.', 'GlaxoSmithKline', 'GlaxoSmithKline Intellectual Property (No. 2) Limited England', 'GlaxoSmithKline Intellectual Property Development Ltd. England', 'GlaxoSmithKline LLC', 'Glax

In [15]:
review_names = sorted(sample.loc[sample["screen"] == "review", "Applicant"].unique())
print(len(review_names), "companies to review")
print(review_names)

206 companies to review
['ADC Therapeutics S.A.', 'AMAG Pharmaceuticals', 'Ablynx NV', 'Acadia Pharmaceuticals', 'Acadia Pharmaceuticals, Inc.', 'Achaogen', 'Actelion Pharmaceuticals', 'Advanced Accelerator Applications', 'Advanced Accelerator Applications USA, Inc.', 'Aerie Pharmaceuticals', 'Aeterna Zentaris GmBH', 'Agios Pharmaceuticals', 'Agios Pharmaceuticals, Inc.', 'Akebia Therapeutics, Inc.', 'Albireo AB', 'Alkermes', 'Alkermes, Inc.', 'Allecra Therapeutics SAS', 'Alnylam Pharmaceuticals', 'Alnylam Pharmaceuticals, Inc.', 'Altor BioScience, LLC', 'Amicus Therapeutics', 'Amicus Therapeutics US, LLC', 'Amryt Pharmaceuticals DAC', 'Amylyx Pharmaceuticals, Inc.', 'Anacor Pharmacueticals', 'Apellis Pharmaceuticals, Inc.', 'Ardea Biosciences', 'Ardelyx', 'Ariad Pharmaceuticals', 'Array Biopharma', 'Arrowhead Pharmaceuticals, Inc.', 'Ascendis Pharma Bone Diseases A/S', 'Ascendis Pharma Endocrinology Division A/S', 'Asklepion Pharmaceuticals', 'Astex Pharmaceuticals, Inc.', 'Athenex, I

In [16]:
sample.to_csv("../data/processed/nme_2015_2025_screened.csv", index=False)

cols = ["Proprietary Name", "Active Ingredient/Moiety", "Applicant",
        "FDA Approval Date", "Approval Year", "Review Designation",
        "Orphan Drug Designation", "Breakthrough Therapy Designation",
        "Fast Track Designation", "Accelerated Approval"]
review = sample[sample["screen"] == "review"][cols]
review = review.sort_values("FDA Approval Date", ascending=False)

for c in ["ticker", "exchange", "market_cap_usd_m", "press_release_datetime_et",
          "timing", "event_day0", "confounding_event", "include", "exclude_reason", "notes"]:
    review[c] = ""

review.to_csv("../data/processed/review_sheet.csv", index=False)
print(len(review), "drugs to research")

226 drugs to research
